# Add project root to path so we can import from parent directory
import sys
import os
from pathlib import Path

# Try multiple strategies to find the project root
def find_project_root():
    # Strategy 1: Look for private_path_query_utils.py going up from cwd
    path = Path.cwd()
    for _ in range(5):  # Check up to 5 levels
        if (path / "private_path_query_utils.py").exists():
            return path
        path = path.parent
    
    # Strategy 2: Use parent of notebooks directory
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    
    # Strategy 3: Hardcoded fallback
    return Path("/Users/joshuamayhugh/Projects/aima-python")

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Also change working directory to project root for consistency
os.chdir(project_root)
print(f"Project root: {project_root}")
print(f"Working directory: {Path.cwd()}")

In [5]:
# Import all necessary modules
from private_path_query_utils import (
    Vertex, 
    Edge, 
    EdgeState, 
    Graph, 
    GraphPath,
    PrivatePathInfo,  # Shared config object
    normalize_edge_weights,
)
from private_path_query_logic import (
    build_bob_q, 
    build_alice_q, 
    build_physics_q,
    build_q,
    ordered_symbols, 
    directed_pairs_from_edges,
    num_bits,
    PosBit,
    Active,
    Allowed,
    compute_hard_clause_weight,  # Dynamic hard weight based on num_parties
)
import numpy as np

print("Imports successful!")


Imports successful!


In [6]:
# =============================================================================
# Step 1: Build a small example graph
# =============================================================================
# 
# Graph structure (V=4 vertices):
#
#     0 -----> 1 -----> 3
#              |
#              v (BLOCKED)
#              2 -----> 3
#
# Edge states: 
#   - 0->1: TRAVERSABLE
#   - 1->2: BLOCKED (cannot use)
#   - 1->3: TRAVERSABLE  
#   - 2->3: TRAVERSABLE

V = 4  # Number of vertices
vertices = [Vertex(i) for i in range(V)]

# Define edges with their states
edges = [
    Edge(vertices[0], vertices[1], EdgeState.TRAVERSABLE),
    Edge(vertices[1], vertices[2], EdgeState.BLOCKED),     # This edge is blocked!
    Edge(vertices[1], vertices[3], EdgeState.TRAVERSABLE),
    Edge(vertices[2], vertices[3], EdgeState.TRAVERSABLE),
]

print(f"Graph with V={V} vertices")
print(f"Number of bits needed: b = ceil(log2({V})) = {num_bits(V)}")
print("\nDirected edges:")
for e in edges:
    state_name = {0: "NO_EDGE", 1: "BLOCKED", 2: "TRAVERSABLE"}[int(e.state)]
    print(f"  {e.vertex1.id} -> {e.vertex2.id}: {state_name}")

Graph with V=4 vertices
Number of bits needed: b = ceil(log2(4)) = 2

Directed edges:
  0 -> 1: TRAVERSABLE
  1 -> 2: BLOCKED
  1 -> 3: TRAVERSABLE
  2 -> 3: TRAVERSABLE


In [7]:
# =============================================================================
# Step 2: Define the path-finding problem and PrivatePathInfo
# =============================================================================

start = 0  # Start vertex
goal = 3   # Goal vertex
T = 2      # Time horizon (number of steps)
num_parties = 2  # Alice + 1 Bob

print(f"Path-finding problem:")
print(f"  Start vertex: {start}")
print(f"  Goal vertex:  {goal}")
print(f"  Time horizon: T = {T}")
print(f"  Num parties:  {num_parties} (Alice + {num_parties - 1} Bob)")
print(f"  Valid path:   0 -> 1 -> 3 (uses traversable edges only)")
print(f"  Invalid path: 0 -> 1 -> 2 -> 3 (edge 1->2 is BLOCKED)")

# Set up the edge domain (directed pairs that exist in the graph)
compact_pairs = directed_pairs_from_edges(edges, V)
print(f"\nEdge domain (directed pairs): {compact_pairs}")

# For PrivatePathInfo, we need UNKNOWN state edges (public domain, not Bob's private states)
# This is the "edge domain" that all parties agree on - the graph structure, not the edge states
unknown_domain_edges = [
    Edge(Vertex(u), Vertex(v), EdgeState.UNKNOWN)
    for (u, v) in compact_pairs
]

# Calculate row counts by building Q matrices (needed for PrivatePathInfo validation)
syms = ordered_symbols(T=T, V=V, directed_pairs=compact_pairs)
b = num_bits(V)

# Temporarily build to get row counts
from private_path_query_logic import _build_alice_q, _build_bob_q
_temp_q_alice, _ = _build_alice_q(start=0, goal=0, T=T, V=V, symbols=syms)
_temp_q_bob, _ = _build_bob_q(edges=[], T=T, V=V, symbols=syms, directed_pairs=compact_pairs)
_temp_q_physics, _ = build_physics_q(T=T, V=V, symbols=syms, directed_pairs=compact_pairs)

alice_rows = int(_temp_q_physics.shape[0] + _temp_q_alice.shape[0])
bob_rows = int(_temp_q_bob.shape[0])

# Create PrivatePathInfo - the shared configuration object
info = PrivatePathInfo(
    num_parties=num_parties,
    T=T,
    V=V,
    rows_per_id=[alice_rows, bob_rows],  # Alice + Physics rows, Bob rows
    use_edge_domain=True,
    edge_domain_edges=unknown_domain_edges,  # Must use UNKNOWN state
)

print(f"\n=== PrivatePathInfo ===")
print(f"  num_parties: {info.num_parties}")
print(f"  T: {info.T}, V: {info.V}")
print(f"  use_edge_domain: {info.use_edge_domain}")
print(f"  edge_domain_pairs: {info.edge_domain_pairs()}")
print(f"  rows_per_id: {info.rows_per_id} (Alice+Physics={alice_rows}, Bob={bob_rows})")
print(f"  hard_clause_weight: {compute_hard_clause_weight(info.num_parties)}")

# Symbol ordering was already computed above for row calculation
print(f"\n=== Symbol ordering ({len(syms)} symbols) ===")
print(f"Position bits: PosBit(t, k) for t in 0..{T}, k in 0..{b-1}")
for t in range(T + 1):
    bits = [f"PosBit({t},{k})" for k in range(b)]
    print(f"  Time {t}: {bits}")

print(f"\nEdge variables:")
active_syms = [s for s in syms if str(s).startswith("Active")]
allowed_syms = [s for s in syms if str(s).startswith("Allowed")]
print(f"  Active:  {active_syms}")
print(f"  Allowed: {allowed_syms}")

Path-finding problem:
  Start vertex: 0
  Goal vertex:  3
  Time horizon: T = 2
  Num parties:  2 (Alice + 1 Bob)
  Valid path:   0 -> 1 -> 3 (uses traversable edges only)
  Invalid path: 0 -> 1 -> 2 -> 3 (edge 1->2 is BLOCKED)

Edge domain (directed pairs): [(0, 1), (1, 2), (1, 3), (2, 3)]

=== PrivatePathInfo ===
  num_parties: 2
  T: 2, V: 4
  use_edge_domain: True
  edge_domain_pairs: [(0, 1), (1, 2), (1, 3), (2, 3)]
  rows_per_id: [52, 12] (Alice+Physics=52, Bob=12)
  hard_clause_weight: 3.0

=== Symbol ordering (14 symbols) ===
Position bits: PosBit(t, k) for t in 0..2, k in 0..1
  Time 0: ['PosBit(0,0)', 'PosBit(0,1)']
  Time 1: ['PosBit(1,0)', 'PosBit(1,1)']
  Time 2: ['PosBit(2,0)', 'PosBit(2,1)']

Edge variables:
  Active:  [Active(0, 1), Active(1, 2), Active(1, 3), Active(2, 3)]
  Allowed: [Allowed(0, 1), Allowed(1, 2), Allowed(1, 3), Allowed(2, 3)]


In [8]:
# =============================================================================
# Step 3: Build Alice's Q matrix (start/goal constraints)
# =============================================================================
#
# Alice provides:
#   - pos(0) = start  (initial position)
#   - pos(T) = goal   (final position)
#
# With bit encoding, these become unit clauses on PosBit variables.

print("=== Alice's Constraints ===")
print(f"Alice wants: pos(0) = {start} and pos(T) = pos({T}) = {goal}")
print()

# Show bit encoding of start and goal
print(f"Bit encoding (b={b} bits):")
print(f"  start={start} in binary: {bin(start)[2:].zfill(b)} -> ", end="")
for k in range(b):
    bit_val = (start >> k) & 1
    lit = f"PosBit(0,{k})" if bit_val else f"~PosBit(0,{k})"
    print(lit, end=" ")
print()

print(f"  goal={goal} in binary:  {bin(goal)[2:].zfill(b)} -> ", end="")
for k in range(b):
    bit_val = (goal >> k) & 1
    lit = f"PosBit({T},{k})" if bit_val else f"~PosBit({T},{k})"
    print(lit, end=" ")
print()

# Build Alice Q matrix using PrivatePathInfo
# Note: hard_clause_weight is automatically computed from info.num_parties
q_alice, w_alice = build_alice_q(
    info=info,  # PrivatePathInfo object
    start=start,
    goal=goal,
    symbols=syms,
    print_cnf_clauses=True,
)

hard_weight = compute_hard_clause_weight(info.num_parties)
print(f"\nQ_alice shape: {q_alice.shape}")
print(f"  - {q_alice.shape[0]} clauses (rows)")
print(f"  - {q_alice.shape[1]} columns = 2 * {len(syms)} symbols (positive + negative literals)")
print(f"  - hard_clause_weight = {hard_weight} (num_parties + 1 = {info.num_parties} + 1)")

=== Alice's Constraints ===
Alice wants: pos(0) = 0 and pos(T) = pos(2) = 3

Bit encoding (b=2 bits):
  start=0 in binary: 00 -> ~PosBit(0,0) ~PosBit(0,1) 
  goal=3 in binary:  11 -> PosBit(2,0) PosBit(2,1) 
=== build_alice_q: Alice physics CNF clauses ===
alice_physics[0] = ~PosBit(0, 0)
  cnf[0] (from local 0, weight=3.0) = ~PosBit(0, 0)
alice_physics[1] = ~PosBit(0, 1)
  cnf[1] (from local 0, weight=3.0) = ~PosBit(0, 1)
alice_physics[2] = PosBit(2, 0)
  cnf[2] (from local 0, weight=3.0) = PosBit(2, 0)
alice_physics[3] = PosBit(2, 1)
  cnf[3] (from local 0, weight=3.0) = PosBit(2, 1)
Total CNF clauses: 4

Q_alice shape: (4, 28)
  - 4 clauses (rows)
  - 28 columns = 2 * 14 symbols (positive + negative literals)
  - hard_clause_weight = 3.0 (num_parties + 1 = 2 + 1)


In [9]:
# =============================================================================
# Step 4: Build Bob's Q matrix (edge state constraints)
# =============================================================================
#
# Bob provides the edge states:
#   - For TRAVERSABLE edges: Active(u,v) AND Allowed(u,v)
#   - For BLOCKED edges:     Active(u,v) AND ~Allowed(u,v)  [soft clause!]
#   - For NO_EDGE:           ~Active(u,v) AND ~Allowed(u,v)
#
# The ~Allowed(u,v) clauses for BLOCKED edges are SOFT constraints (weight < 1)
# meaning they can be violated if necessary to find a satisfying assignment.
#
# Hard clause weight = num_parties + 1 (e.g., 3.0 for Alice + 1 Bob)

print("=== Bob's Edge Constraints ===")
print("Bob knows the state of each edge:")
for e in edges:
    state_name = {0: "NO_EDGE", 1: "BLOCKED", 2: "TRAVERSABLE"}[int(e.state)]
    u, v = e.vertex1.id, e.vertex2.id
    if int(e.state) == 2:  # TRAVERSABLE
        print(f"  Edge {u}->{v} {state_name}: Active({u},{v})=T, Allowed({u},{v})=T")
    elif int(e.state) == 1:  # BLOCKED
        print(f"  Edge {u}->{v} {state_name}:  Active({u},{v})=T, Allowed({u},{v})=F [SOFT]")
    else:
        print(f"  Edge {u}->{v} {state_name}:    Active({u},{v})=F, Allowed({u},{v})=F")
print()

# Build Bob Q matrix using PrivatePathInfo
# Note: edge weights are automatically normalized and hard_clause_weight is computed
q_bob, w_bob = build_bob_q(
    info=info,  # PrivatePathInfo object
    edges=edges,
    symbols=syms,
    print_cnf_clauses=True,
    normalize_weights=True,  # Auto-normalize soft clause weights to sum to 1.0
)

hard_weight = compute_hard_clause_weight(info.num_parties)
print(f"\nQ_bob shape: {q_bob.shape}")
print(f"Clause weights: {w_bob}")
print(f"\nNote: weight < {hard_weight} clauses are SOFT (can be violated)")
print(f"      weight = {hard_weight} clauses are HARD (must be satisfied)")

=== Bob's Edge Constraints ===
Bob knows the state of each edge:
  Edge 0->1 TRAVERSABLE: Active(0,1)=T, Allowed(0,1)=T
  Edge 1->2 BLOCKED:  Active(1,2)=T, Allowed(1,2)=F [SOFT]
  Edge 1->3 TRAVERSABLE: Active(1,3)=T, Allowed(1,3)=T
  Edge 2->3 TRAVERSABLE: Active(2,3)=T, Allowed(2,3)=T

=== build_bob_q: Bob physics CNF clauses ===
bob_physics[0] = Active(0, 1)
  cnf[0] (from local 0, weight=3.0) = Active(0, 1)
bob_physics[1] = Allowed(0, 1)
  cnf[1] (from local 0, weight=3.0) = Allowed(0, 1)
bob_physics[2] = (Allowed(0, 1) ==> Active(0, 1))
  cnf[2] (from local 0, weight=3.0) = (Active(0, 1) | ~Allowed(0, 1))
bob_physics[3] = Active(1, 2)
  cnf[3] (from local 0, weight=3.0) = Active(1, 2)
bob_physics[4] = ~Allowed(1, 2)
  cnf[4] (from local 0, weight=0.25) = ~Allowed(1, 2)
bob_physics[5] = (Allowed(1, 2) ==> Active(1, 2))
  cnf[5] (from local 0, weight=3.0) = (Active(1, 2) | ~Allowed(1, 2))
bob_physics[6] = Active(1, 3)
  cnf[6] (from local 0, weight=3.0) = Active(1, 3)
bob_physics[7

In [10]:
# =============================================================================
# Step 5: Build Physics Q matrix (transition constraints)
# =============================================================================
#
# Physics constraints encode:
#   1. Range restriction: forbid invalid vertex codes (for V not power of 2)
#   2. Transition feasibility: pos(t)=u -> OR_{v in adj[u]} pos(t+1)=v
#   3. Edge legality: (pos(t)=u AND pos(t+1)=v) -> Allowed(u,v)
#   4. Edge consistency: Allowed(u,v) -> Active(u,v)
#
# All physics clauses are HARD (weight = num_parties + 1)

print("=== Physics Constraints ===")
print(f"With V={V} and b={b} bits, valid codes are 0..{V-1}")
if V < (1 << b):
    invalid_codes = list(range(V, 1 << b))
    print(f"Invalid codes to forbid: {invalid_codes}")
else:
    print("V is a power of 2, no invalid codes to forbid")
print()

# Build Physics Q matrix with dynamic hard clause weight
hard_weight = compute_hard_clause_weight(info.num_parties)
q_physics, w_physics = build_physics_q(
    T=info.T,
    V=info.V,
    symbols=syms,
    directed_pairs=compact_pairs,
    print_cnf_clauses=True,
    hard_clause_weight=hard_weight,  # Uses num_parties + 1
)

print(f"\nQ_physics shape: {q_physics.shape}")
print(f"  - {q_physics.shape[0]} clauses encoding graph dynamics")
print(f"  - hard_clause_weight = {hard_weight}")

=== Physics Constraints ===
With V=4 and b=2 bits, valid codes are 0..3
V is a power of 2, no invalid codes to forbid

=== build_physics_q: Physics CNF clauses ===
physics[0] = ((~PosBit(0, 0) & ~PosBit(0, 1)) ==> ((PosBit(1, 0) & ~PosBit(1, 1)) | (~PosBit(1, 0) & ~PosBit(1, 1))))
  cnf[0] (from local 0, weight=3.0) = (~PosBit(1, 0) | PosBit(1, 0) | PosBit(0, 0) | PosBit(0, 1))
  cnf[1] (from local 1, weight=3.0) = (~PosBit(1, 1) | PosBit(1, 0) | PosBit(0, 0) | PosBit(0, 1))
  cnf[2] (from local 2, weight=3.0) = (~PosBit(1, 0) | ~PosBit(1, 1) | PosBit(0, 0) | PosBit(0, 1))
  cnf[3] (from local 3, weight=3.0) = (~PosBit(1, 1) | ~PosBit(1, 1) | PosBit(0, 0) | PosBit(0, 1))
physics[1] = ((PosBit(0, 0) & ~PosBit(0, 1)) ==> (((~PosBit(1, 0) & PosBit(1, 1)) | (PosBit(1, 0) & PosBit(1, 1))) | (PosBit(1, 0) & ~PosBit(1, 1))))
  cnf[4] (from local 0, weight=3.0) = (PosBit(1, 0) | PosBit(1, 0) | ~PosBit(1, 0) | ~PosBit(0, 0) | PosBit(0, 1))
  cnf[5] (from local 1, weight=3.0) = (~PosBit(1, 1) | 

In [11]:
# =============================================================================
# Step 6: Combine into full Q matrix
# =============================================================================
#
# The full Q matrix is the vertical concatenation of:
#   1. Physics clauses (graph dynamics)
#   2. Alice clauses (start/goal)
#   3. Bob clauses (edge states)
#
# In MPC, Alice contributes physics + her constraints, Bob contributes his constraints.

print("=== Full Q Matrix ===")
print(f"Q_physics: {q_physics.shape[0]} rows")
print(f"Q_alice:   {q_alice.shape[0]} rows")
print(f"Q_bob:     {q_bob.shape[0]} rows")

# Stack them
q_full = np.vstack([q_physics, q_alice, q_bob])
w_full = np.concatenate([w_physics, w_alice, w_bob])

hard_weight = compute_hard_clause_weight(info.num_parties)
print(f"\nFull Q shape: {q_full.shape}")
print(f"  - {q_full.shape[0]} total clauses")
print(f"  - {q_full.shape[1]} columns = 2 * {len(syms)} (pos + neg literals)")
print(f"\nWeight vector shape: {w_full.shape}")
print(f"  Hard clauses (weight={hard_weight}): {np.sum(w_full == hard_weight)}")
print(f"  Soft clauses (weight<{hard_weight}): {np.sum(w_full < hard_weight)}")
print(f"\nHard weight formula: num_parties + 1 = {info.num_parties} + 1 = {hard_weight}")

=== Full Q Matrix ===
Q_physics: 48 rows
Q_alice:   4 rows
Q_bob:     12 rows

Full Q shape: (64, 28)
  - 64 total clauses
  - 28 columns = 2 * 14 (pos + neg literals)

Weight vector shape: (64,)
  Hard clauses (weight=3.0): 63
  Soft clauses (weight<3.0): 1

Hard weight formula: num_parties + 1 = 2 + 1 = 3.0


In [12]:
# =============================================================================
# Step 7: Understand the Q matrix structure
# =============================================================================
#
# Each row of Q encodes one CNF clause.
# Column layout: [positive literals | negative literals]
#   - Columns 0..n-1: positive literals (x_i appears in clause)
#   - Columns n..2n-1: negative literals (~x_i appears in clause)
#
# Example: clause (x_0 OR ~x_2) with n=3 symbols
#   -> row = [1, 0, 0, 0, 0, 1]
#            pos:x0 pos:x1 pos:x2 neg:x0 neg:x1 neg:x2

n = len(syms)
print(f"=== Q Matrix Structure ===")
print(f"Number of symbols: n = {n}")
print(f"Q has {q_full.shape[1]} columns = 2 * {n}")
print(f"  Columns 0..{n-1}: positive literals")
print(f"  Columns {n}..{2*n-1}: negative literals")
print()

# Show symbol mapping
print("Symbol index mapping:")
for i, sym in enumerate(syms):
    print(f"  {i}: {sym} (positive col={i}, negative col={n+i})")

print()
print("Example: First Alice clause (start constraint)")
alice_row = q_alice[0]
pos_lits = [syms[i] for i in range(n) if alice_row[i] == 1]
neg_lits = [syms[i] for i in range(n) if alice_row[n + i] == 1]
print(f"  Positive literals: {pos_lits}")
print(f"  Negative literals: {neg_lits}")
print(f"  Clause: {' OR '.join([str(l) for l in pos_lits] + [f'~{l}' for l in neg_lits])}")

=== Q Matrix Structure ===
Number of symbols: n = 14
Q has 28 columns = 2 * 14
  Columns 0..13: positive literals
  Columns 14..27: negative literals

Symbol index mapping:
  0: PosBit(0, 0) (positive col=0, negative col=14)
  1: PosBit(0, 1) (positive col=1, negative col=15)
  2: PosBit(1, 0) (positive col=2, negative col=16)
  3: PosBit(1, 1) (positive col=3, negative col=17)
  4: PosBit(2, 0) (positive col=4, negative col=18)
  5: PosBit(2, 1) (positive col=5, negative col=19)
  6: Active(0, 1) (positive col=6, negative col=20)
  7: Active(1, 2) (positive col=7, negative col=21)
  8: Active(1, 3) (positive col=8, negative col=22)
  9: Active(2, 3) (positive col=9, negative col=23)
  10: Allowed(0, 1) (positive col=10, negative col=24)
  11: Allowed(1, 2) (positive col=11, negative col=25)
  12: Allowed(1, 3) (positive col=12, negative col=26)
  13: Allowed(2, 3) (positive col=13, negative col=27)

Example: First Alice clause (start constraint)
  Positive literals: []
  Negative lite

In [13]:
# =============================================================================
# Step 8: Summary - Bit encoding efficiency
# =============================================================================

print("=== Summary: Bit Encoding vs One-Hot ===")
print()
print(f"Problem size: V={info.V} vertices, T={info.T} timesteps, {len(compact_pairs)} edges")
print(f"Parties: {info.num_parties} (Alice + {info.num_parties - 1} Bob)")
print()

# Compare variable counts
old_at_vars = (info.T + 1) * info.V
old_move_vars = info.T * info.V * (info.V - 1)  # dense
old_wait_vars = info.T * info.V
old_total = old_at_vars + old_move_vars + old_wait_vars

new_pos_vars = (info.T + 1) * b
new_edge_vars = 2 * len(compact_pairs)  # Active + Allowed
new_total = new_pos_vars + new_edge_vars

print("Old one-hot encoding (dense):")
print(f"  At(t,v):     {old_at_vars}")
print(f"  Move(t,u,v): {old_move_vars}")  
print(f"  Wait(t,v):   {old_wait_vars}")
print(f"  Total:       {old_at_vars + old_move_vars + old_wait_vars}")
print()
print("New bit encoding (sparse edge domain):")
print(f"  PosBit(t,k): {new_pos_vars}")
print(f"  Active/Allowed: {new_edge_vars}")
print(f"  Total:       {new_total}")
print()
print(f"Variable reduction: {old_total} -> {new_total} ({100*(1 - new_total/old_total):.1f}% fewer)")
print()
print("Key insight: AMO (at-most-one) is FREE with bit encoding!")
print("  - Old: O(V²) pairwise clauses per timestep")
print("  - New: 0 clauses (binary arithmetic guarantees uniqueness)")
print()
print("=== Dynamic Hard Clause Weight ===")
hard_weight = compute_hard_clause_weight(info.num_parties)
print(f"hard_weight = num_parties + 1 = {info.num_parties} + 1 = {hard_weight}")
print("This ensures hard clauses always dominate soft clauses from all parties.")

=== Summary: Bit Encoding vs One-Hot ===

Problem size: V=4 vertices, T=2 timesteps, 4 edges
Parties: 2 (Alice + 1 Bob)

Old one-hot encoding (dense):
  At(t,v):     12
  Move(t,u,v): 24
  Wait(t,v):   8
  Total:       44

New bit encoding (sparse edge domain):
  PosBit(t,k): 6
  Active/Allowed: 8
  Total:       14

Variable reduction: 44 -> 14 (68.2% fewer)

Key insight: AMO (at-most-one) is FREE with bit encoding!
  - Old: O(V²) pairwise clauses per timestep
  - New: 0 clauses (binary arithmetic guarantees uniqueness)

=== Dynamic Hard Clause Weight ===
hard_weight = num_parties + 1 = 2 + 1 = 3.0
This ensures hard clauses always dominate soft clauses from all parties.


## Recap: How the Q Matrix is Built

### PrivatePathInfo - Shared Configuration
All parties share a `PrivatePathInfo` object containing:
- `num_parties`: Number of parties (Alice + Bobs)
- `T`: Time horizon (number of steps)
- `V`: Number of vertices
- `use_edge_domain`: Whether to use sparse edge representation
- `edge_domain_edges`: List of valid (u, v) pairs

### Alice's Contribution
- **Start constraint**: Unit clauses fixing `pos(0) = start`
- **Goal constraint**: Unit clauses fixing `pos(T) = goal`
- Each vertex is encoded as `b = ceil(log2(V))` bits

### Bob's Contribution  
- **TRAVERSABLE edge (u,v)**: `Active(u,v)` AND `Allowed(u,v)`
- **BLOCKED edge (u,v)**: `Active(u,v)` AND `~Allowed(u,v)` (soft clause!)
- **NO_EDGE**: `~Active(u,v)` AND `~Allowed(u,v)`
- Edge weights can be assigned to control which blocked edges are relaxed first

### Physics Constraints
- **Range restriction**: Forbid invalid vertex codes >= V
- **Transition feasibility**: `pos(t)=u -> OR_{v in adj[u]} pos(t+1)=v`
- **Edge legality**: `(pos(t)=u AND pos(t+1)=v) -> Allowed(u,v)`
- **Consistency**: `Allowed(u,v) -> Active(u,v)`

### Clause Weights
- **Hard clauses**: weight = `num_parties + 1` (must be satisfied)
- **Soft clauses**: weight < hard weight (can be relaxed if needed)
- Dynamic hard weight ensures hard clauses always dominate

### Q Matrix Layout
```
Q = | Q_physics |   (graph dynamics)
    | Q_alice   |   (start/goal)
    | Q_bob     |   (edge states)
```

Each row is a CNF clause, columns are `[positive literals | negative literals]`.